# Lab 07 — Joins and Broadcast

**Tujuan:** Membandingkan shuffle join dan broadcast join dengan physical plan.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 07")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

## 1. Generate transaction and dimension data

In [ ]:
transactions = spark.range(0, 1_000_000, 1, 4)
transactions = transactions.withColumn("customer_id", F.col("id") % 10000)
customers = spark.range(0, 10000, 1, 2).withColumnRenamed("id", "customer_id")

## 2. Force comparison to be fair

Matikan auto broadcast sementara agar bisa melihat join tanpa broadcast otomatis.

In [ ]:
old_threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
regular = transactions.join(customers, "customer_id")
regular.explain("formatted")
print("Rows:", regular.count())

## 3. Explicit broadcast

Bandingkan operator join dan `Exchange`.

In [ ]:
broadcast_join = transactions.join(F.broadcast(customers), "customer_id")
broadcast_join.explain("formatted")
print("Rows:", broadcast_join.count())
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", old_threshold)

**Tugas:** jelaskan kapan broadcast dimension table bisa menghemat shuffle, dan mengapa broadcast tabel sangat besar justru berbahaya.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.